In [ ]:
!nvidia-smi


Fri Sep 18 13:00:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

REPO_URL = "https://github.com/siifat/green-llm-token-research"
PROJECT = "/content/drive/MyDrive/green-llm-token-research"

if not os.path.exists(PROJECT):
    !git clone "$REPO_URL" "$PROJECT"
else:
    print("Project folder already exists. Not cloning again.")

%cd "$PROJECT"

Cloning into '/content/drive/MyDrive/green-llm-token-research'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 98 (delta 4), reused 98 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 439.01 KiB | 7.84 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/drive/MyDrive/green-llm-token-research


In [ ]:
from pathlib import Path
import csv

csv_path = Path("green_llm_ready_bundle/green_llm_ready_100_prompts.csv")

print("Dataset exists:", csv_path.exists())

with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

print("Number of prompt pairs:", len(rows))

p3 = [r for r in rows if r["prompt_id"] == "P0003"][0]
print("P0003 source pair:", p3["source_pair_id"])
print("P0003 task:", p3["task"])

Dataset exists: True
Number of prompt pairs: 100
P0003 source pair: 1.26
P0003 task: Write the discussion of a net present value appraisal for a finance assignment.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (350 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os
import subprocess
import time

os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)
print("Ollama server started.")

Ollama server started.


In [ ]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [ ]:
!ollama pull llama3.2:3b-instruct-q4_K_M

In [ ]:
!ollama run llama3.2:3b-instruct-q4_K_M "Reply with exactly: READY"

READY



In [ ]:
!ollama ps

NAME                           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.2:3b-instruct-q4_K_M    a80c4f17acd5    2.6 GB    100% GPU     4096       4 minutes from now    


In [ ]:
from pathlib import Path

source = Path("scripts/run_experiment.py")
target = Path("scripts/run_experiment_llama32_colab.py")

text = source.read_text(encoding="utf-8")

text = text.replace(
    'MODEL = "gemma3:4b"',
    'MODEL = "llama3.2:3b-instruct-q4_K_M"'
)

text = text.replace(
    'output_path = project_root / "data" / "raw" / "runs.jsonl"',
    'output_path = project_root / "data" / "raw" / "runs_llama32_3b_colab.jsonl"'
)

text = text.replace(
    'error_path = project_root / "logs" / "experiment_errors.jsonl"',
    'error_path = project_root / "logs" / "experiment_errors_llama32_3b_colab.jsonl"'
)

target.write_text(text, encoding="utf-8")

print("Created:", target)

Created: scripts/run_experiment_llama32_colab.py


In [ ]:
!grep -n 'MODEL =' scripts/run_experiment_llama32_colab.py

11:MODEL = "llama3.2:3b-instruct-q4_K_M"


In [ ]:
!python scripts/run_experiment_llama32_colab.py --limit 1

Model: llama3.2:3b-instruct-q4_K_M
Tasks in scope: 1
Expected runs: 2
Already completed: 0
Remaining: 2
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_llama32_3b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/1] P0001 - baseline ... done | in=108 out=331 total=439 time=4.75s
[1/1] P0001 - optimized ... done | in=76 out=289 total=365 time=4.34s

Run pass finished.
Completed: 2/2
Still missing: 0

All experimental runs completed successfully.


In [ ]:
from pathlib import Path

for p in [
    Path("data/raw/runs_llama32_3b_colab.jsonl"),
    Path("logs/experiment_errors_llama32_3b_colab.jsonl"),
]:
    if p.exists():
        p.unlink()
        print("Deleted pilot file:", p)

Deleted pilot file: data/raw/runs_llama32_3b_colab.jsonl


In [ ]:
!python scripts/run_experiment_llama32_colab.py

Model: llama3.2:3b-instruct-q4_K_M
Tasks in scope: 100
Expected runs: 200
Already completed: 0
Remaining: 200
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_llama32_3b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/100] P0001 - baseline ... done | in=108 out=331 total=439 time=5.30s
[1/100] P0001 - optimized ... done | in=76 out=289 total=365 time=4.85s
[2/100] P0002 - optimized ... done | in=90 out=468 total=558 time=7.94s
[2/100] P0002 - baseline ... done | in=105 out=604 total=709 time=11.15s
[3/100] P0003 - baseline ... done | in=103 out=294 total=397 time=5.59s
[3/100] P0003 - optimized ... done | in=82 out=276 total=358 time=5.03s
[4/100] P0004 - optimized ... done | in=68 out=184 total=252 time=3.26s
[4/100] P0004 - baseline ... done | in=89 out=234 total=323 time=4.08s
[5/100] P0005 - baseline ... done | in=88 out=197 total=285 time=3.33s
[5/100] P0005 - optimized ... done | in=76 out=165 total=241 time=2.88s
[6/100] P0006 - optimized ... done

In [ ]:
import json
from pathlib import Path

result_file = Path("data/raw/runs_llama32_3b_colab.jsonl")

records = [
    json.loads(line)
    for line in result_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Total records:", len(records))

Total records: 200


In [ ]:
from collections import Counter

print(Counter(r["variant"] for r in records))

Counter({'baseline': 100, 'optimized': 100})


In [ ]:
from collections import Counter

keys = [(r["prompt_id"], r["variant"]) for r in records]
counts = Counter(keys)

bad = {k: v for k, v in counts.items() if v != 1}

print("Unique prompt/variant records:", len(counts))
print("Problems:", bad)

Unique prompt/variant records: 200
Problems: {}


In [ ]:
from collections import defaultdict

pairs = defaultdict(dict)

for r in records:
    pairs[r["prompt_id"]][r["variant"]] = r

baseline_input = sum(p["baseline"]["input_tokens"] for p in pairs.values())
optimized_input = sum(p["optimized"]["input_tokens"] for p in pairs.values())

baseline_output = sum(p["baseline"]["output_tokens"] for p in pairs.values())
optimized_output = sum(p["optimized"]["output_tokens"] for p in pairs.values())

baseline_total = sum(p["baseline"]["total_tokens"] for p in pairs.values())
optimized_total = sum(p["optimized"]["total_tokens"] for p in pairs.values())

baseline_time = sum(p["baseline"]["wall_time_s"] for p in pairs.values())
optimized_time = sum(p["optimized"]["wall_time_s"] for p in pairs.values())

def reduction(b, o):
    return (b - o) / b * 100

fewer_total = sum(
    p["optimized"]["total_tokens"] < p["baseline"]["total_tokens"]
    for p in pairs.values()
)

faster = sum(
    p["optimized"]["wall_time_s"] < p["baseline"]["wall_time_s"]
    for p in pairs.values()
)

print("=== LLAMA 3.2 3B SUMMARY ===")
print(f"Input-token reduction: {reduction(baseline_input, optimized_input):.2f}%")
print(f"Output-token reduction: {reduction(baseline_output, optimized_output):.2f}%")
print(f"Total-token reduction: {reduction(baseline_total, optimized_total):.2f}%")
print(f"Wall-time reduction: {reduction(baseline_time, optimized_time):.2f}%")
print(f"Optimized used fewer total tokens in: {fewer_total}/100 pairs")
print(f"Optimized was faster in: {faster}/100 pairs")

=== LLAMA 3.2 3B SUMMARY ===
Input-token reduction: 53.32%
Output-token reduction: -63.46%
Total-token reduction: -25.07%
Wall-time reduction: -57.18%
Optimized used fewer total tokens in: 86/100 pairs
Optimized was faster in: 73/100 pairs


In [ ]:
from google.colab import files
files.download('data/raw/runs_llama32_3b_colab.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Folder location:", PROJECT)

Folder location: /content/drive/MyDrive/green-llm-token-research


In [ ]:
from google.colab import files
files.download('data/raw/runs_llama32_3b_colab.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>